# P12 Length Defense Reproducible

由原 P12 拆出：檢查效能提升是否只由文字長度增加造成。


## 0. Setup


In [1]:
from __future__ import annotations

from pathlib import Path
from statistics import mean, median, stdev
from collections import Counter, defaultdict
from datetime import datetime
import csv
import json
import math
import re

try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except Exception:
    scipy_stats = None
    SCIPY_AVAILABLE = False

BASE = Path("/home/ester/valid_description")
TGT = BASE / "experiment data" / "journal_figures_tables_20260629"
FIG = TGT / "figures"
TAB = TGT / "tables"
DER = TGT / "derived_audits"
SRC = TGT / "source_programs"
REPO = BASE / "text-conditioned-outfit-recommendation"
EXP_MAIN = REPO / "experiments"
EXP_NONE = REPO / "experiments_none_only"

for p in [FIG, TAB, DER, SRC]:
    p.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
WORD_RE = re.compile(r"[A-Za-z]+(?:[-'][A-Za-z]+)?")

def read_json(path: Path):
    with path.open(encoding="utf-8") as f:
        return json.load(f)

def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        raise ValueError(f"No rows to write: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print("[write]", path)

def display_csv(path: Path, max_rows: int = 12) -> None:
    rows = read_csv(path)
    for r in rows[:max_rows]:
        print(r)
    if len(rows) > max_rows:
        print(f"... ({len(rows) - max_rows} more rows)")

def fmt_float(x, nd=4) -> str:
    try:
        if x is None or math.isnan(float(x)):
            return ""
        return f"{float(x):.{nd}f}"
    except Exception:
        return ""

def fmt_pct(num, den) -> str:
    return f"{100 * float(num) / float(den):.2f}" if den else "0.00"

def to_int(x, default=0) -> int:
    try:
        return int(float(x))
    except Exception:
        return default

def to_float(x, default=0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default

def query_token_count(text: str) -> int:
    return len(TOKEN_RE.findall(str(text or "").strip()))

def word_tokens(text: str) -> list[str]:
    return [t.lower() for t in WORD_RE.findall(str(text or "").strip())]

def word_count(text: str) -> int:
    return len(word_tokens(text))

def original_field(original_data: dict, set_id: str, field: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get(field, "") or "").strip()
    return str(value or "").strip()

def original_url_name_plus_title(original_data: dict, set_id: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        parts = [str(value.get("url_name", "") or "").strip(), str(value.get("title", "") or "").strip()]
        return " ".join(p for p in parts if p).strip()
    return str(value or "").strip()

def generated_title_only(generated_data: dict, set_id: str) -> str:
    value = generated_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get("title", "") or "").strip()
    return str(value or "").strip()

print("BASE =", BASE)
print("TAB  =", TAB)
print("DER  =", DER)
print("SCIPY_AVAILABLE =", SCIPY_AVAILABLE)


BASE = /home/ester/valid_description
TAB  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables
DER  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits
SCIPY_AVAILABLE = True


## 2. Length Defense: Is the Gain Only Due to Longer Text?


In [3]:
def load_cir_detail(tag: str) -> dict[tuple[str, str, str], dict[str, str]]:
    out = {}
    for fp in sorted(EXP_MAIN.glob(f"detail_cir_{tag}_seed*.csv")):
        for r in read_csv(fp):
            out[(str(r["seed"]), str(r["set_id"]), str(r["target_item_id"]))] = r
    return out

def pearson_p(xs, ys):
    if len(xs) < 3 or len(set(xs)) <= 1 or len(set(ys)) <= 1:
        return None, None
    if SCIPY_AVAILABLE:
        o = scipy_stats.pearsonr(xs, ys)
        return float(o.statistic), float(o.pvalue)
    mx, my = mean(xs), mean(ys)
    vx, vy = sum((x - mx) ** 2 for x in xs), sum((y - my) ** 2 for y in ys)
    if vx == 0 or vy == 0:
        return None, None
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / (vx * vy) ** 0.5, None

def spearman_p(xs, ys):
    if len(xs) < 3 or len(set(xs)) <= 1 or len(set(ys)) <= 1:
        return None, None
    if SCIPY_AVAILABLE:
        o = scipy_stats.spearmanr(xs, ys)
        return float(o.statistic), float(o.pvalue)
    return None, None

def bucket_delta(n):
    if n <= 0: return "01_delta_le_0"
    if n <= 4: return "02_delta_1_4"
    if n <= 8: return "03_delta_5_8"
    if n <= 12: return "04_delta_9_12"
    return "05_delta_13_plus"

def bucket_gen(n):
    if n <= 8: return "01_gen_len_0_8"
    if n <= 12: return "02_gen_len_9_12"
    if n <= 16: return "03_gen_len_13_16"
    return "04_gen_len_17_plus"

def bucket_orig(n):
    if n <= 3: return "01_original_len_0_3"
    if n <= 6: return "02_original_len_4_6"
    if n <= 9: return "03_original_len_7_9"
    return "04_original_len_10_plus"

def summarize_perf(rows, label):
    if not rows:
        return {"group": label, "n_seed_rows": "0", "n_unique_set_target_pairs": "0", "original_hit10": "", "full_hit10": "", "delta_hit10": "", "original_mean_rank": "", "full_mean_rank": "", "mean_rank_improvement": "", "mean_original_query_len": "", "mean_generated_query_len": "", "mean_length_delta": ""}
    oh = [to_int(r["original_hit10"]) for r in rows]
    fh = [to_int(r["full_hit10"]) for r in rows]
    orank = [to_int(r["original_rank"]) for r in rows]
    frank = [to_int(r["full_rank"]) for r in rows]
    olen = [to_int(r["original_query_len_tokens"]) for r in rows]
    glen = [to_int(r["generated_query_len_tokens"]) for r in rows]
    dlen = [to_int(r["length_delta_tokens"]) for r in rows]
    return {
        "group": label,
        "n_seed_rows": str(len(rows)),
        "n_unique_set_target_pairs": str(len({(r["set_id"], r["target_item_id"]) for r in rows})),
        "original_hit10": fmt_float(mean(oh), 4),
        "full_hit10": fmt_float(mean(fh), 4),
        "delta_hit10": fmt_float(mean([f - o for f, o in zip(fh, oh)]), 4),
        "original_mean_rank": fmt_float(mean(orank), 2),
        "full_mean_rank": fmt_float(mean(frank), 2),
        "mean_rank_improvement": fmt_float(mean([o - f for o, f in zip(orank, frank)]), 2),
        "mean_original_query_len": fmt_float(mean(olen), 2),
        "mean_generated_query_len": fmt_float(mean(glen), 2),
        "mean_length_delta": fmt_float(mean(dlen), 2),
    }

original = read_json(BASE / "polyvore_data/polyvore_outfits/polyvore_outfit_titles.json")
generated = read_json(BASE / "new_polyvore_outfit_titles.json")
old = load_cir_detail("cir_old_featold")
new = load_cir_detail("cir_new_featnew")
keys = sorted(set(old) & set(new), key=lambda x: (int(x[0]), int(x[1]), int(x[2])))

rowlevel = []
for seed, sid, iid in keys:
    o, f = old[(seed, sid, iid)], new[(seed, sid, iid)]
    olen = query_token_count(original_url_name_plus_title(original, sid))
    glen = query_token_count(generated_title_only(generated, sid))
    orank, frank = to_int(o["rank"]), to_int(f["rank"])
    oh, fh = to_int(o["hit@10"]), to_int(f["hit@10"])
    rowlevel.append({
        "scope": "main_experiments_original_like",
        "seed": seed, "set_id": sid, "target_item_id": iid, "target_item_fg": str(o.get("target_item_fg", "")),
        "original_query_len_tokens": str(olen), "generated_query_len_tokens": str(glen), "length_delta_tokens": str(glen - olen),
        "generated_len_bucket": bucket_gen(glen), "length_delta_bucket": bucket_delta(glen - olen), "original_len_bucket": bucket_orig(olen),
        "original_rank": str(orank), "full_rank": str(frank), "rank_improvement": str(orank - frank),
        "original_hit10": str(oh), "full_hit10": str(fh), "hit10_delta": str(fh - oh),
    })
write_csv(DER / "A07_length_performance_rowlevel_cir.csv", rowlevel)

pair_acc = defaultdict(lambda: {"length_delta_tokens": 0.0, "original_query_len_tokens": 0.0, "generated_query_len_tokens": 0.0, "hit10_delta": [], "rank_improvement": []})
for r in rowlevel:
    d = pair_acc[(r["set_id"], r["target_item_id"])]
    d["length_delta_tokens"] = float(r["length_delta_tokens"])
    d["original_query_len_tokens"] = float(r["original_query_len_tokens"])
    d["generated_query_len_tokens"] = float(r["generated_query_len_tokens"])
    d["hit10_delta"].append(float(r["hit10_delta"]))
    d["rank_improvement"].append(float(r["rank_improvement"]))
pair_rows = []
for (sid, iid), d in pair_acc.items():
    pair_rows.append({"set_id": sid, "target_item_id": iid, "length_delta_tokens": d["length_delta_tokens"], "original_query_len_tokens": d["original_query_len_tokens"], "generated_query_len_tokens": d["generated_query_len_tokens"], "hit10_delta": mean(d["hit10_delta"]), "rank_improvement": mean(d["rank_improvement"])})

corr_rows = []
for scope, rows in [("seed_row_level", rowlevel), ("pair_mean_across_seeds", pair_rows)]:
    for xname in ["original_query_len_tokens", "generated_query_len_tokens", "length_delta_tokens"]:
        for yname in ["hit10_delta", "rank_improvement"]:
            xs = [float(r[xname]) for r in rows]
            ys = [float(r[yname]) for r in rows]
            pr, pp = pearson_p(xs, ys)
            sr, sp = spearman_p(xs, ys)
            abs_pr = abs(pr or 0)
            corr_rows.append({
                "scope": scope, "x": xname, "y": yname, "n": str(len(rows)),
                "pearson_r": fmt_float(pr, 6), "pearson_p": fmt_float(pp, 6),
                "spearman_r": fmt_float(sr, 6), "spearman_p": fmt_float(sp, 6),
                "practical_effect_size": "negligible" if abs_pr < 0.05 else ("small" if abs_pr < 0.10 else "inspect"),
                "interpretation_hint": "Large N can make tiny correlations statistically significant; use effect size plus bucket/matched checks.",
            })
write_csv(DER / "A08_length_performance_correlation_summary.csv", corr_rows)

bucket_rows = []
for label, field in [("generated_query_length_bucket", "generated_len_bucket"), ("length_delta_bucket", "length_delta_bucket"), ("original_query_length_bucket", "original_len_bucket")]:
    for b in sorted(set(r[field] for r in rowlevel)):
        bucket_rows.append({"bucket_type": label, **summarize_perf([r for r in rowlevel if r[field] == b], b)})
write_csv(DER / "A09_length_bucket_performance_summary.csv", bucket_rows)

matched = []
for th in [0, 1, 2, 3, 5]:
    subset = [r for r in rowlevel if abs(to_int(r["length_delta_tokens"])) <= th]
    matched.append({"matching_rule": f"abs(generated_query_len - original_query_len) <= {th} tokens", **summarize_perf(subset, f"abs_delta_le_{th}")})
write_csv(DER / "A10_matched_length_subset_summary.csv", matched)

hit = next(r for r in corr_rows if r["scope"] == "pair_mean_across_seeds" and r["x"] == "length_delta_tokens" and r["y"] == "hit10_delta")
rank = next(r for r in corr_rows if r["scope"] == "pair_mean_across_seeds" and r["x"] == "length_delta_tokens" and r["y"] == "rank_improvement")
lines = [
    "Length defense audit",
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    "Scope:",
    "- Main CIR experiment only: experiments/detail_cir_cir_old_featold_seed*.csv vs detail_cir_cir_new_featnew_seed*.csv.",
    "- Original query follows url_name + title; generated query uses generated title only.",
    "- No new descriptions, no no-image rewrite baseline, and no retraining are introduced here.",
    "",
    "How to answer the length-confound question:",
    "- The method effect and the length-confound test are different hypotheses.",
    "- The proposed method may be statistically better than the original baseline.",
    "- If improvement were merely caused by longer text, length_delta should have a meaningful association with Hit@10 or rank improvement.",
    "- With thousands of rows, p-values can be significant even when the correlation is practically tiny; report r/effect size together with p-value, bucket analysis, and matched-length subsets.",
    "",
    "Pair-mean correlation across the 9,311 CIR query-target pairs:",
    f"- length_delta_tokens vs Hit@10 improvement: Pearson r={hit['pearson_r']}, p={hit['pearson_p']}; Spearman r={hit['spearman_r']}, p={hit['spearman_p']}.",
    f"- length_delta_tokens vs rank improvement: Pearson r={rank['pearson_r']}, p={rank['pearson_p']}; Spearman r={rank['spearman_r']}, p={rank['spearman_p']}.",
    "",
    "Writing guidance:",
    "- Do not write that length has absolutely no statistical signal if p-values are significant under large N.",
    "- Write that the association is practically negligible and is not sufficient to explain the retrieval gains.",
    "- A true masked-query performance claim requires a separate masked-query CIR rerun.",
]
(DER / "A11_length_defense_interpretation.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

display_csv(DER / "A08_length_performance_correlation_summary.csv", max_rows=20)


[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A07_length_performance_rowlevel_cir.csv
[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A08_length_performance_correlation_summary.csv
[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A09_length_bucket_performance_summary.csv
[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A10_matched_length_subset_summary.csv
{'scope': 'seed_row_level', 'x': 'original_query_len_tokens', 'y': 'hit10_delta', 'n': '46555', 'pearson_r': '-0.010763', 'pearson_p': '0.020223', 'spearman_r': '-0.011066', 'spearman_p': '0.016955', 'practical_effect_size': 'negligible', 'interpretation_hint': 'Large N can make tiny correlations statistically significant; use effect size plus bucket/matched checks.'}
{'scope': 'seed_row_level', 'x': 'original_query_len_token